# gVCF to VDS

In [1]:
%%configure -f
{
    "driverMemory": "45G"
}

In [2]:
# Import and initiate HAIL
import hail as hl
hl.init(sc)

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
0,application_1744853935852_0001,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

pip-installed Hail requires additional configuration options in Spark referring
  to the path to the Hail Python module directory HAIL_DIR,
  e.g. /path/to/python/site-packages/hail:
    spark.jars=HAIL_DIR/backend/hail-all-spark.jar
    spark.driver.extraClassPath=HAIL_DIR/backend/hail-all-spark.jar
    spark.executor.extraClassPath=./hail-all-spark.jarRunning on Apache Spark version 3.5.2-amzn-1
SparkUI available at http://ip-192-168-66-228.ap-southeast-1.compute.internal:42729
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.134-952ae203dbbe
LOGGING: writing to /mnt/yarn/usercache/livy/appcache/application_1744853935852_0001/container_1744853935852_0001_01_000001/hail-20250417-0158-0.2.134-952ae203dbbe.log

## Load 1KG gVCF into VDS

- List the single sample hard-filtered.gvcf.gz generated for 1000 genomes dragen 3.7.6 analysis
- Upload the list (csv file) into S3
- Set gvcf_list_path

In [3]:
gvcf_list_path='s3://npm-grids/hebrardms/batch/sg10k_reprocess_gvcf_manifest.csv'
vds_prefix = 's3://precise-scratch/hebrardms/SG10K_Health/VDS'

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [4]:
# Import csv samples to ht
ht_gvcf = hl.import_table(gvcf_list_path, delimiter=',', quote = '"', no_header=True)
# Rename the columns
ht_gvcf = ht_gvcf.rename({'f0': 'bucket', 'f1': 'prefix'})
# Build S3 path
ht_gvcf = ht_gvcf.annotate(
    s3_path = hl.str('s3://') + hl.str(ht_gvcf.bucket) + '/' + hl.str(ht_gvcf.prefix)
)

ht_gvcf.count()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

10322
2025-04-17 02:00:20.338 Hail: INFO: Reading table without type imputation
  Loading field 'f0' as type str (not specified)
  Loading field 'f1' as type str (not specified)

In [5]:
# List of S3 path
ls_gvcf = ht_gvcf.s3_path.collect()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
## Test
###
# gvcf_paths=ls_gvcf[0:10] # variant_data: 13905139 rows and 10 columns in 2586 partitions ~ 30Gb ~ 2h on 9 CPU onDemand

In [9]:
# Combine gVCF
combiner = hl.vds.new_combiner(
    output_path=f'{vds_prefix}/SG10K_Health_100.vds',
    temp_path=f'{vds_prefix}/checkpoints/',
    gvcf_paths=ls_gvcf[0:10],
    use_genome_default_intervals=True,
    reference_genome='GRCh38' 
)

combiner.run()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

2025-04-17 02:06:57.156 Hail: WARN: expected input file 's3://precise-wgs-databundle/byob/sg10k-wgs-dragen/WHB6375/6de5907f-6049-4852-99c7-adfda64d6889/output/try-1/WHB6375.hard-filtered.gvcf.gz' to end in .vcf[.bgz, .gz]
2025-04-17 02:06:57.326 Hail: WARN: expected input file 's3://precise-wgs-databundle/byob/sg10k-wgs-dragen/WHB6375/6de5907f-6049-4852-99c7-adfda64d6889/output/try-1/WHB6375.hard-filtered.gvcf.gz' to end in .vcf[.bgz, .gz]
2025-04-17 02:07:06.017 Hail: WARN: generated combiner save path of s3://precise-scratch/hebrardms/SG10K_Health/VDS/checkpoints/combiner-plans/vds-combiner-plan_67729d69425a0e088bda131a531945b342f3fc053fb581ee8220b96bf272efb0_0.2.134.json
2025-04-17 02:07:06.053 Hail: INFO: Running VDS combiner:
    VDS arguments: 0 datasets with 0 samples
    GVCF arguments: 10 inputs/samples
    Branch factor: 100
    GVCF merge batch size: 50
2025-04-17 02:07:06.386 Hail: INFO: GVCF combine (job 1): merging 10 GVCFs into 1 datasets
2025-04-17 03:08:01.718 Hail: IN